# Solving the 8-Puzzle Using Heuristic Search
## Hill Climbing, Best First Search and A* for an Informed Agent

### Practical Implementation in Python

**Objective:** Build an AI agent that solves the 8-Puzzle using **heuristics**. Implement **Hill Climbing**, **Best First Search** and **A\* Search**, then compare all three on:

1. Solution quality
2. Number of explored states
3. Execution time

> This notebook is designed to be easy for students to understand. Run the cells from top to bottom.

# 1. Learning Objectives

After completing this practical, you should be able to:

- Explain the difference between **uninformed** and **informed** search.
- Understand what a **heuristic** is and what makes one admissible.
- Implement the **Misplaced Tiles** heuristic.
- Implement the **Manhattan Distance** heuristic.
- Explain what a **Priority Queue** is and why A\* uses it.
- Implement Hill Climbing from scratch.
- Implement Best First Search from scratch.
- Implement A\* Search from scratch.
- Explain why Hill Climbing gets stuck at a **local minimum**.
- Identify when A\* is better and when Hill Climbing is useful.

# 2. Problem Statement

The **8-Puzzle** is a 3x3 board holding eight numbered tiles and one blank space.

A tile next to the blank space can slide into it. By sliding tiles one at a time,
we must turn the starting board into the goal board.

```text
   START              GOAL

  1  3  6           1  2  3
  5  _  2    -->    4  5  6
  4  7  8           7  8  _
```

Our task is:

> **Find a sequence of moves that turns the START board into the GOAL board.**

In Practical 1 the agent had no idea which location was closer to the goal. Here
it will be able to *estimate* how close each board is, and use that estimate to
choose what to try next.

# 3. What is a State Space?

A **state** is one complete arrangement of the board.

- The **start state** is the board we begin with.
- The **goal state** is the board we want.
- A **move** takes us from one state to another state.

```text
              1 3 6
              5 _ 2      <- current state
              4 7 8
             /  |  \
        Up  /  Left \  Right
           /    |     \
      1 _ 6   1 3 6    1 3 6
      5 3 2   _ 5 2    5 2 _
      4 7 8   4 7 8    4 7 8
```

Every arrangement reachable from the start makes up the **state space**. The
8-Puzzle has 181,440 reachable states, which is far too many to look at one by
one — so the agent needs a way to decide which state to try next.

We will store a board as a **tuple of 9 numbers**, reading left to right, top to
bottom. `0` represents the blank space.

In [1]:
# A board is a tuple of 9 numbers, read left to right, top to bottom.
# 0 is the blank space.

START = (1, 3, 6,
         5, 0, 2,
         4, 7, 8)

GOAL = (1, 2, 3,
        4, 5, 6,
        7, 8, 0)


def show(state):
    """Print a board in 3 rows."""
    for row in range(3):
        line = state[row * 3: row * 3 + 3]
        print(" ".join(str(x) if x != 0 else "_" for x in line))


print("START board:")
show(START)

print()
print("GOAL board:")
show(GOAL)

START board:
1 3 6
5 _ 2
4 7 8

GOAL board:
1 2 3
4 5 6
7 8 _


# 4. What is a Heuristic?

In Practical 1, BFS and DFS were **uninformed**. They had no idea whether they
were getting closer to the goal, so they simply explored everything in a fixed
order.

A **heuristic** is a rule that estimates how far a state is from the goal.

```text
   Uninformed search              Informed search

   "Try everything in order"      "This board looks 4 moves away.
                                   That one looks 9 moves away.
                                   Try the 4 first."
```

We write the heuristic as **h(state)**:

- `h(state) = 0` means we are at the goal.
- A **small** h means the state looks close to the goal.
- A **large** h means the state looks far away.

> Important: a heuristic is a *guess*. It does not have to be exactly right. But
> a good heuristic should never **overestimate** the true distance. A heuristic
> that never overestimates is called **admissible**, and admissibility is what
> makes A\* guaranteed to find the shortest solution.

We will build two heuristics and compare them.

# 5. Heuristic 1: Misplaced Tiles

The simplest heuristic. Count how many tiles are **not** in their goal position.

```text
   Current            Goal          Tile in place?

  1  3  6           1  2  3         1 yes
  5  _  2    vs     4  5  6         3 no
  4  7  8           7  8  _         6 no ... and so on
```

We do **not** count the blank space, because the blank is not a tile we need to
place.

If 7 tiles are out of position, then we need at least 7 moves, because one move
can only put one tile in place. So this heuristic never overestimates — it is
admissible.

In [2]:
def h_misplaced(state):
    """Count tiles that are not in their goal position (ignore the blank)."""
    count = 0
    for i in range(9):
        if state[i] != 0 and state[i] != GOAL[i]:
            count = count + 1
    return count


print("START board:")
show(START)
print()
print("Misplaced tiles h(START) =", h_misplaced(START))
print("Misplaced tiles h(GOAL)  =", h_misplaced(GOAL))

START board:
1 3 6
5 _ 2
4 7 8

Misplaced tiles h(START) = 7
Misplaced tiles h(GOAL)  = 0


# 6. Heuristic 2: Manhattan Distance

Counting misplaced tiles ignores **how far** each tile has to travel. A tile one
square from home counts the same as a tile in the opposite corner.

The **Manhattan Distance** heuristic fixes this. For every tile, count how many
rows and columns it is away from its goal position, then add all of those up.

```text
   Tile 6 is here          Tile 6 belongs here

   1  3 [6]                1  2  3
   5  _  2                 4  5 [6]
   4  7  8                 7  8  _

   Row difference    = 1
   Column difference = 0
   Distance for tile 6 = 1 + 0 = 1
```

It is called Manhattan Distance because you may only move along rows and columns,
like walking city blocks — never diagonally.

This is also admissible, and it is **always at least as large** as the misplaced
count, which makes it the better of the two.

In [3]:
def h_manhattan(state):
    """Total row + column distance of every tile from its goal position."""
    total = 0
    for i in range(9):
        tile = state[i]
        if tile == 0:
            continue

        goal_index = GOAL.index(tile)

        current_row, current_col = i // 3, i % 3
        goal_row, goal_col = goal_index // 3, goal_index % 3

        distance = abs(current_row - goal_row) + abs(current_col - goal_col)
        total = total + distance
    return total


print("START board:")
show(START)
print()
print("Misplaced tiles    h(START) =", h_misplaced(START))
print("Manhattan distance h(START) =", h_manhattan(START))
print()
print("Manhattan is larger, so it is the better estimate.")
print()
print("The true answer for this board is 8 moves.")
print("Misplaced tiles says 7  -> an underestimate, which is allowed.")
print("Manhattan says 8        -> exactly right, which is also allowed.")
print()
print("An admissible heuristic may equal the true cost.")
print("It must simply never go ABOVE it.")

START board:
1 3 6
5 _ 2
4 7 8

Misplaced tiles    h(START) = 7
Manhattan distance h(START) = 8

Manhattan is larger, so it is the better estimate.

The true answer for this board is 8 moves.
Misplaced tiles says 7  -> an underestimate, which is allowed.
Manhattan says 8        -> exactly right, which is also allowed.

An admissible heuristic may equal the true cost.
It must simply never go ABOVE it.


# 7. Generating Moves

Before any algorithm can search, it needs to know which boards can be reached
from the current board in one move.

Only the blank space moves. It can slide **Up**, **Down**, **Left** or **Right**,
as long as it does not fall off the board.

```text
   blank in the middle        blank in a corner
   -> 4 possible moves        -> 2 possible moves

      1  3  6                    1  3  6
      5 [_] 2                    5  2  8
      4  7  8                    4  7 [_]
```

In [4]:
def get_moves(state):
    """Return a list of (new_state, move_name) reachable in one slide."""
    results = []

    blank = state.index(0)
    row, col = blank // 3, blank % 3

    directions = [(-1, 0, "Up"), (1, 0, "Down"), (0, -1, "Left"), (0, 1, "Right")]

    for row_change, col_change, name in directions:
        new_row = row + row_change
        new_col = col + col_change

        # Stay on the board
        if 0 <= new_row < 3 and 0 <= new_col < 3:
            target = new_row * 3 + new_col

            # Swap the blank with the tile next to it
            board = list(state)
            board[blank], board[target] = board[target], board[blank]

            results.append((tuple(board), name))

    return results


print("From the START board we can reach", len(get_moves(START)), "boards:\n")

for new_state, move_name in get_moves(START):
    print("Move blank", move_name, " -> Manhattan h =", h_manhattan(new_state))
    show(new_state)
    print()

From the START board we can reach 4 boards:

Move blank Up  -> Manhattan h = 9
1 _ 6
5 3 2
4 7 8

Move blank Down  -> Manhattan h = 9
1 3 6
5 7 2
4 _ 8

Move blank Left  -> Manhattan h = 7
1 3 6
_ 5 2
4 7 8

Move blank Right  -> Manhattan h = 7
1 3 6
5 2 _
4 7 8



# 8. Priority Queue Before A*

In Practical 1, BFS used a **Queue** (first in, first out) and DFS used a
**Stack** (last in, first out). Neither looks at *how good* an item is.

Informed search needs something different: always take out the **best** item,
whatever order it went in. That structure is a **Priority Queue**.

```text
Items go in with a score:

   ("Board A", 9)   ("Board B", 4)   ("Board C", 7)

Coming out, the SMALLEST score always leaves first:

   Board B (4)  ->  Board C (7)  ->  Board A (9)
```

In Python we use the `heapq` module on a normal list:

- `heappush()` adds an item.
- `heappop()` removes the item with the smallest score.

> Remember the pattern from Practical 1:
>
> **BFS = Queue = FIFO**, **DFS = Stack = LIFO**, **A\* = Priority Queue = best first**

In [5]:
import heapq

priority_queue = []

# Each item is a (score, name) pair. heapq sorts on the first value.
heapq.heappush(priority_queue, (9, "Board A"))
heapq.heappush(priority_queue, (4, "Board B"))
heapq.heappush(priority_queue, (7, "Board C"))

print("Items added: Board A (9), Board B (4), Board C (7)")
print()

print("Removing items one at a time:")
while priority_queue:
    score, name = heapq.heappop(priority_queue)
    print("  removed", name, "with score", score)

Items added: Board A (9), Board B (4), Board C (7)

Removing items one at a time:
  removed Board B with score 4
  removed Board C with score 7
  removed Board A with score 9


# 9. Hill Climbing

Hill Climbing is the simplest informed strategy.

**Look at all the boards you can reach in one move. Go to the best one. Repeat.**

```text
h = 8   ->   h = 7   ->   h = 6   ->   h = 5   ->   h = 4   ->   ???
start                                                        all neighbours
                                                             are WORSE (h = 5)
```

It never looks back and it never keeps a list of alternatives. It only ever holds
the board it is standing on.

## Why Hill Climbing gets stuck

If **every** neighbour is worse than the current board, Hill Climbing has nowhere
to go. It stops — even though it is not at the goal.

This is called a **local minimum**: a state that looks best compared with
everything next to it, while a much better state exists further away.

```text
        Getting stuck

   h=5      h=5
      \    /
       \  /
        h=4      <- we are here, and both neighbours are worse
                    so we stop, but h is not 0
```

> Important: Hill Climbing is fast and uses almost no memory, but it does **not**
> guarantee that it will find the goal at all.

## Hill Climbing Algorithm

1. Start at the start board.
2. Look at every board reachable in one move.
3. Find the neighbour with the smallest h value.
4. If that neighbour is better than the current board, move to it.
5. If no neighbour is better, stop and report being stuck.
6. Repeat until the goal is reached or we get stuck.

In [6]:
def hill_climbing(start, goal, heuristic):
    current = start
    path = [start]
    explored = [start]

    while True:
        print("Standing on a board with h =", heuristic(current))

        # Goal test
        if current == goal:
            return path, explored, "Goal reached"

        # Find the best neighbour
        best_state = None
        best_value = heuristic(current)

        for new_state, move_name in get_moves(current):
            value = heuristic(new_state)
            if value < best_value:
                best_state = new_state
                best_value = value

        # No neighbour is an improvement
        if best_state is None:
            return None, explored, "Stuck at a local minimum"

        current = best_state
        path.append(current)
        explored.append(current)

    return None, explored, "Unknown"

# 10. Run Hill Climbing

We will use the Manhattan Distance heuristic.

In [7]:
print("===== HILL CLIMBING EXECUTION =====\n")

hc_path, hc_explored, hc_status = hill_climbing(START, GOAL, h_manhattan)

print("\n===== HILL CLIMBING RESULT =====")
print("Status:", hc_status)
print("Explored states:", len(hc_explored))

print()
print("The board it stopped on:")
show(hc_explored[-1])
print()
print("h of this board =", h_manhattan(hc_explored[-1]), " (a goal would be 0)")
print()
print("Its neighbours:")
for new_state, move_name in get_moves(hc_explored[-1]):
    print("  Move", move_name, "-> h =", h_manhattan(new_state))

===== HILL CLIMBING EXECUTION =====

Standing on a board with h = 8
Standing on a board with h = 7
Standing on a board with h = 6
Standing on a board with h = 5
Standing on a board with h = 4

===== HILL CLIMBING RESULT =====
Status: Stuck at a local minimum
Explored states: 5

The board it stopped on:
1 3 6
4 5 2
7 8 _

h of this board = 4  (a goal would be 0)

Its neighbours:
  Move Up -> h = 5
  Move Left -> h = 5


Read that output carefully. Hill Climbing made four good moves, dropping h from
9 down to 4 — and then stopped.

The board it stopped on is **not** the goal, and every single move available from
it makes h go **up** to 5. There is no downhill step left to take, so the
algorithm has nowhere to go.

This is the local minimum problem, and it is the reason we need algorithms that
remember alternatives.

# 11. Best First Search

Best First Search fixes the main weakness of Hill Climbing: it **keeps a list of
every board it has seen but not yet explored**, in a priority queue.

So when one route stops looking good, it can jump back to a promising board it
noticed earlier. It can never get stuck the way Hill Climbing does.

It chooses the next board using h alone:

```text
   Pick the board with the smallest h.
   "Which board LOOKS closest to the goal?"
```

## Why this is called greedy

Best First Search only asks how far it still has to go. It completely ignores how
far it has already travelled — so it will happily take a long, winding route if
each individual step looked good at the time.

> Important: Best First Search will usually find a solution, but it does **not**
> guarantee the shortest one.

## Best First Search Algorithm

1. Put the starting path into the priority queue with score h(start).
2. Remove the path with the smallest score.
3. Check the current board.
4. If it is the goal, return the path.
5. Otherwise, add all unvisited neighbouring boards, scored by h.
6. Repeat until the goal is found.

In [8]:
def best_first_search(start, goal, heuristic):
    # Each item is (score, path). heapq removes the smallest score first.
    queue = [(heuristic(start), [start])]

    visited = set()
    explored = []

    while queue:
        score, path = heapq.heappop(queue)
        current = path[-1]

        if current in visited:
            continue

        visited.add(current)
        explored.append(current)

        # Goal test
        if current == goal:
            return path, explored

        # Add neighbouring boards, scored by h only
        for new_state, move_name in get_moves(current):
            if new_state not in visited:
                heapq.heappush(queue, (heuristic(new_state), path + [new_state]))

    return None, explored

# 12. Run Best First Search

In [9]:
print("===== BEST FIRST SEARCH EXECUTION =====\n")

bf_path, bf_explored = best_first_search(START, GOAL, h_manhattan)

print("Explored", len(bf_explored), "boards. First 10, in order:\n")
for i, state in enumerate(bf_explored[:10]):
    flat = " ".join(str(x) if x != 0 else "_" for x in state)
    print("  %2d. %s   h = %d" % (i + 1, flat, h_manhattan(state)))

print("\n===== BEST FIRST SEARCH RESULT =====")
print("Solution found:", bf_path is not None)
print("Number of moves:", len(bf_path) - 1)
print("Explored states:", len(bf_explored))

===== BEST FIRST SEARCH EXECUTION =====

Explored 25 boards. First 10, in order:

   1. 1 3 6 5 _ 2 4 7 8   h = 8
   2. 1 3 6 _ 5 2 4 7 8   h = 7
   3. 1 3 6 4 5 2 _ 7 8   h = 6
   4. 1 3 6 4 5 2 7 _ 8   h = 5
   5. 1 3 6 4 5 2 7 8 _   h = 4
   6. 1 3 6 4 5 _ 7 8 2   h = 5
   7. 1 3 _ 4 5 6 7 8 2   h = 4
   8. 1 _ 3 4 5 6 7 8 2   h = 3
   9. _ 1 3 4 5 6 7 8 2   h = 4
  10. 1 5 3 4 _ 6 7 8 2   h = 4

===== BEST FIRST SEARCH RESULT =====
Solution found: True
Number of moves: 10
Explored states: 25


It found a solution, and it never got stuck. But look at the number of moves — it
is more than the 8 moves we know are possible.

That is the price of being greedy. Each step looked like the best move *at that
moment*, and nobody was keeping track of how long the journey was becoming.

# 13. A* Search

A\* fixes exactly that. It scores a board using **both** halves of the journey:

```text
     f(board)  =  g(board)  +  h(board)

                     |            |
        moves already made    estimated moves
        to get here           still to go
```

- **g** is the real cost so far — how many moves we have actually made.
- **h** is the estimated cost remaining — our heuristic.
- **f** is the estimated total length of the whole solution.

A\* always expands the board with the smallest **f**, which means it prefers a
board that is both *cheap to have reached* and *close to the goal*.

## Why A* finds the shortest solution

Because h never overestimates, f never overestimates the length of the best
solution passing through that board. So A\* cannot finish through a long route
while a shorter one is still waiting in the queue.

> **A\* = Priority Queue ordered by f = g + h**
>
> If the heuristic is admissible, A\* is guaranteed to find the shortest solution.

## A* Algorithm

1. Put the starting path into the priority queue with score f = 0 + h(start).
2. Remove the path with the smallest f.
3. Check the current board.
4. If it is the goal, return the path.
5. Otherwise, for each neighbour compute g + 1 and f = (g + 1) + h, and add it.
6. Repeat until the goal is found.

In [10]:
def a_star_search(start, goal, heuristic):
    # Each item is (f, g, path)
    queue = [(heuristic(start), 0, [start])]

    visited = set()
    explored = []

    while queue:
        f, g, path = heapq.heappop(queue)
        current = path[-1]

        if current in visited:
            continue

        visited.add(current)
        explored.append(current)

        # Goal test
        if current == goal:
            return path, explored

        # Every move costs 1, so the next g is g + 1
        for new_state, move_name in get_moves(current):
            if new_state not in visited:
                new_g = g + 1
                new_f = new_g + heuristic(new_state)
                heapq.heappush(queue, (new_f, new_g, path + [new_state]))

    return None, explored

# 14. Run A*

In [11]:
print("===== A* SEARCH EXECUTION =====\n")

astar_path, astar_explored = a_star_search(START, GOAL, h_manhattan)

print("Explored", len(astar_explored), "boards, in order:\n")
for i, state in enumerate(astar_explored):
    flat = " ".join(str(x) if x != 0 else "_" for x in state)
    print("  %2d. %s   h = %d" % (i + 1, flat, h_manhattan(state)))

print("\n===== A* RESULT =====")
print("Solution found:", astar_path is not None)
print("Number of moves:", len(astar_path) - 1)
print("Explored states:", len(astar_explored))

===== A* SEARCH EXECUTION =====

Explored 13 boards, in order:

   1. 1 3 6 5 _ 2 4 7 8   h = 8
   2. 1 3 6 _ 5 2 4 7 8   h = 7
   3. 1 3 6 5 2 _ 4 7 8   h = 7
   4. 1 3 6 4 5 2 _ 7 8   h = 6
   5. 1 3 _ 5 2 6 4 7 8   h = 6
   6. 1 3 6 4 5 2 7 _ 8   h = 5
   7. 1 _ 3 5 2 6 4 7 8   h = 5
   8. 1 3 6 4 5 2 7 8 _   h = 4
   9. 1 2 3 5 _ 6 4 7 8   h = 4
  10. 1 2 3 _ 5 6 4 7 8   h = 3
  11. 1 2 3 4 5 6 _ 7 8   h = 2
  12. 1 2 3 4 5 6 7 _ 8   h = 1
  13. 1 2 3 4 5 6 7 8 _   h = 0

===== A* RESULT =====
Solution found: True
Number of moves: 8
Explored states: 13


In [12]:
# The full solution, board by board
print("===== THE SOLUTION A* FOUND =====\n")

for step, state in enumerate(astar_path):
    label = "START" if step == 0 else ("GOAL" if step == len(astar_path) - 1
                                       else "Move %d" % step)
    print(label)
    show(state)
    print()

===== THE SOLUTION A* FOUND =====

START
1 3 6
5 _ 2
4 7 8

Move 1
1 3 6
5 2 _
4 7 8

Move 2
1 3 _
5 2 6
4 7 8

Move 3
1 _ 3
5 2 6
4 7 8

Move 4
1 2 3
5 _ 6
4 7 8

Move 5
1 2 3
_ 5 6
4 7 8

Move 6
1 2 3
4 5 6
_ 7 8

Move 7
1 2 3
4 5 6
7 _ 8

GOAL
1 2 3
4 5 6
7 8 _



# 15. Compare Execution Time

For a fair comparison, we will create quieter versions without print statements.

We use `time.perf_counter()` to measure elapsed execution time.

> Because this puzzle is small, execution times are tiny and can vary between
> runs. The comparison demonstrates **how to measure algorithm performance**, not
> that one algorithm is always faster by a fixed amount.

In [13]:
import time


def hill_climbing_quiet(start, goal, heuristic):
    current = start
    path = [start]
    explored = [start]
    while True:
        if current == goal:
            return path, explored, "Goal reached"
        best_state = None
        best_value = heuristic(current)
        for new_state, move_name in get_moves(current):
            value = heuristic(new_state)
            if value < best_value:
                best_state = new_state
                best_value = value
        if best_state is None:
            return None, explored, "Stuck at a local minimum"
        current = best_state
        path.append(current)
        explored.append(current)


# Best First Search and A* do not print, so we can time them directly.

start_time = time.perf_counter()
hc_path, hc_explored, hc_status = hill_climbing_quiet(START, GOAL, h_manhattan)
hc_time = time.perf_counter() - start_time

start_time = time.perf_counter()
bf_path, bf_explored = best_first_search(START, GOAL, h_manhattan)
bf_time = time.perf_counter() - start_time

start_time = time.perf_counter()
astar_path, astar_explored = a_star_search(START, GOAL, h_manhattan)
astar_time = time.perf_counter() - start_time

print("Hill Climbing Time    :", hc_time, "seconds")
print("Best First Search Time:", bf_time, "seconds")
print("A* Search Time        :", astar_time, "seconds")

Hill Climbing Time    : 0.0002136049999990064 seconds
Best First Search Time: 0.00035301900004469644 seconds
A* Search Time        : 0.0002152860000705914 seconds


# 16. Compare Solution Quality

For this practical, **solution quality** is the number of moves used.

A smaller number of moves is better.

> The true shortest solution for our START board is **8 moves**. A\* should match
> this. Best First Search is not guaranteed to.

In [14]:
hc_moves = len(hc_path) - 1 if hc_path else None
bf_moves = len(bf_path) - 1 if bf_path else None
astar_moves = len(astar_path) - 1 if astar_path else None

print("Hill Climbing moves    :", hc_moves, "(", hc_status, ")")
print("Best First Search moves:", bf_moves)
print("A* Search moves        :", astar_moves)
print()
print("Shortest possible      : 8")

Hill Climbing moves    : None ( Stuck at a local minimum )
Best First Search moves: 10
A* Search moves        : 8

Shortest possible      : 8


# 17. Does a Better Heuristic Help?

We claimed Manhattan Distance is the better heuristic. Let us test that claim by
running the same two searches with each heuristic and counting explored states.

In [15]:
rows = []
for name, heuristic in [("Misplaced Tiles", h_misplaced),
                        ("Manhattan Distance", h_manhattan)]:
    bf_p, bf_e = best_first_search(START, GOAL, heuristic)
    a_p, a_e = a_star_search(START, GOAL, heuristic)
    rows.append((name,
                 len(bf_p) - 1, len(bf_e),
                 len(a_p) - 1, len(a_e)))

print("%-20s %-12s %-10s %-10s %-10s" % ("Heuristic", "BestFirst", "explored",
                                          "A* moves", "explored"))
print("-" * 66)
for name, bfm, bfe, am, ae in rows:
    print("%-20s %-12d %-10d %-10d %-10d" % (name, bfm, bfe, am, ae))

Heuristic            BestFirst    explored   A* moves   explored  
------------------------------------------------------------------
Misplaced Tiles      14           101        8          19        
Manhattan Distance   10           25         8          13        


Two things to notice.

First, **A\* found an 8-move solution with both heuristics.** The heuristic does
not change whether A\* is correct — admissibility guarantees that.

Second, **the better heuristic did less work.** Manhattan Distance explored fewer
boards than Misplaced Tiles in both algorithms. A more accurate estimate means
fewer wrong turns.

> A better heuristic does not make the answer better. It makes the answer
> **cheaper to find**.

# 18. Final Comparison Table

In [16]:
results = {
    "Metric": [
        "Solution Found",
        "Number of Moves",
        "Explored States",
        "Execution Time (seconds)",
        "Data Structure",
        "Uses Cost So Far (g)?",
        "Shortest Solution Guaranteed?"
    ],
    "Hill Climbing": [
        "No - " + hc_status,
        hc_moves if hc_moves else "No solution",
        len(hc_explored),
        hc_time,
        "None (only current state)",
        "No",
        "No"
    ],
    "Best First Search": [
        "Yes",
        bf_moves,
        len(bf_explored),
        bf_time,
        "Priority Queue (h)",
        "No",
        "No"
    ],
    "A* Search": [
        "Yes",
        astar_moves,
        len(astar_explored),
        astar_time,
        "Priority Queue (f = g + h)",
        "Yes",
        "Yes, if h is admissible"
    ]
}

# Display using pandas if available
try:
    import pandas as pd
    comparison = pd.DataFrame(results)
    display(comparison)
except ImportError:
    for i in range(len(results["Metric"])):
        print(results["Metric"][i])
        print("  Hill Climbing    :", results["Hill Climbing"][i])
        print("  Best First Search:", results["Best First Search"][i])
        print("  A* Search        :", results["A* Search"][i])
        print()

,Metric,Hill Climbing,Best First Search,A* Search
0,Solution Found,No - Stuck at a local minimum,Yes,Yes
1,Number of Moves,No solution,10,8
2,Explored States,5,25,13
3,Execution Time (seconds),0.000214,0.000353,0.000215
4,Data Structure,None (only current state),Priority Queue (h),Priority Queue (f = g + h)
5,Uses Cost So Far (g)?,No,No,Yes
6,Shortest Solution Guaranteed?,No,No,"Yes, if h is admissible"


# 19. Heuristic Search — Conceptual Comparison

| Feature | Hill Climbing | Best First Search | A* Search |
|---|---|---|---|
| Score Used | h | h | f = g + h |
| Main Data Structure | None | Priority Queue | Priority Queue |
| Remembers Alternatives | No | Yes | Yes |
| Can Get Stuck | Yes | No | No |
| Counts Moves Already Made | No | No | Yes |
| Shortest Solution | Not guaranteed | Not guaranteed | Guaranteed if h is admissible |
| Memory Requirement | Very low | High | High |
| Best for | Quick approximate answers | Fast, roughly good answers | Shortest solution |

# 20. When is A* Better?

A\* is better when:

1. We need the **shortest solution**, not just any solution.
2. We have a heuristic that never overestimates.
3. We can afford to store the boards waiting in the priority queue.

### Example: Route Planning with Real Distances

In Practical 1 we found routes on an unweighted campus map, where every road
counted as one step. Real maps have real distances.

**Example question:**

> Which route reaches the hospital in the shortest total distance?

Here g is the distance already travelled and h is the straight-line distance
remaining.

**Recommended algorithm: A\***

# 21. When is Hill Climbing Useful?

Hill Climbing can be useful when:

1. The state space is far too large to store a queue of alternatives.
2. A good-enough answer is acceptable and the shortest answer is not required.
3. Memory is very limited.
4. We can afford to restart it several times from different starting points.

### Important Note

Because Hill Climbing keeps only the current state, it uses almost no memory at
all — but as we saw in section 10, it can stop without reaching the goal.

A common fix is **Random Restart Hill Climbing**: run it many times from
different random starts and keep the best result. This does not guarantee the
shortest solution either, but it makes getting stuck far less likely.

# 22. Important Limitation

A\* is guaranteed to find the shortest solution, but that guarantee costs memory.

A\* stores every board waiting to be explored. For a small 8-Puzzle this is fine.
For a 15-Puzzle, or a real map with millions of junctions, the priority queue can
grow until it fills all available memory.

Real systems therefore use variations such as:

- **Iterative Deepening A\*** (IDA\*), which uses much less memory
- **Weighted A\***, which trades a little solution quality for a lot of speed
- **Beam Search**, which keeps only the best few states at each level

There is also one requirement that is easy to forget:

> If your heuristic **overestimates**, A\* is no longer guaranteed to find the
> shortest solution. It will still return an answer, and it will not warn you.

# 23. Student Exercise

Try the following:

### Exercise 1
Change `START` to a different board and run all three algorithms again.

### Exercise 2
Run Hill Climbing with `h_misplaced` instead of `h_manhattan`. Does it still get
stuck? Does it stop on the same board?

### Exercise 3
Write a third heuristic that always returns `0`, and run A\* with it. What
algorithm from Practical 1 does A\* now behave like, and why?

### Exercise 4
Write a heuristic that returns `h_manhattan(state) * 5`. Run A\* with it and
check whether the solution is still 8 moves. Explain what happened.

### Exercise 5
Compare, for every algorithm:
- Solution found
- Number of moves
- Explored states
- Execution time

### Exercise 6
Change the order of the `directions` list inside `get_moves`. Which algorithms
change their answer, and which do not?

In [17]:
# Student Practice Area
# Try a different start board here.

# Example:
# START = (1, 2, 3,
#          4, 5, 6,
#          0, 7, 8)

# Then run:
# path, explored = a_star_search(START, GOAL, h_manhattan)
# print("Moves:", len(path) - 1)
# print("Explored:", len(explored))

print("Practice area ready!")

Practice area ready!


# 24. Conclusion

In this practical, we built an informed search agent for the 8-Puzzle and
compared three heuristic strategies.

### Hill Climbing
- Keeps **only the current state**
- Always moves to the best neighbour
- Uses almost no memory
- **Gets stuck** at a local minimum and may never reach the goal

### Best First Search
- Uses a **Priority Queue** ordered by **h**
- Never gets stuck, because it remembers alternatives
- Greedy: ignores how far it has already travelled
- Does **not** guarantee the shortest solution

### A* Search
- Uses a **Priority Queue** ordered by **f = g + h**
- Counts both the journey so far and the journey remaining
- Guarantees the shortest solution when h is admissible
- Costs the most memory

### Final Decision

For an 8-Puzzle, or any problem where the **shortest solution matters**,
**A\* is generally the better choice**.

Hill Climbing is useful when memory is tight and an approximate answer is
acceptable. Best First Search sits between the two: fast and never stuck, but
without any quality guarantee.

**Key idea:**

> A heuristic tells the agent which way to look. Adding the cost already paid,
> as A\* does with f = g + h, is what turns a good guess into a guarantee.